In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf

In [3]:
# ==========================================================
# Configuration
# ==========================================================

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Archive"

TEST_DIR = os.path.join(BASE_DIR, "test")
DARK_DIR = os.path.join(BASE_DIR, "test_dark")
BLUR_DIR = os.path.join(BASE_DIR, "test_blur")
ROTATE_DIR = os.path.join(BASE_DIR, "test_rotate")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

MODEL_PATHS = {
    "Baseline CNN": os.path.join(BASE_DIR, "CNN_Baseline.keras"),
    "CNN (Data Augmentation)": os.path.join(BASE_DIR, "CNN_Augmentation.keras"),
    "ResNet50": os.path.join(BASE_DIR, "ResNet50.keras"),
    "EfficientNet-B0": os.path.join(BASE_DIR, "EfficientNetB0.keras"),
    "ViT": os.path.join(BASE_DIR, "ViT.keras"),
}

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Create Datasets

In [4]:
def create_transformed_dataset(source_dir, target_dir, transform):
    """Create a transformed copy of a class-organised image dataset."""
    os.makedirs(target_dir, exist_ok=True)

    for class_name in sorted(os.listdir(source_dir)):
        source_class_dir = os.path.join(source_dir, class_name)

        if not os.path.isdir(source_class_dir):
            continue

        target_class_dir = os.path.join(target_dir, class_name)
        os.makedirs(target_class_dir, exist_ok=True)

        for image_name in sorted(os.listdir(source_class_dir)):
            source_path = os.path.join(source_class_dir, image_name)
            target_path = os.path.join(target_class_dir, image_name)

            image = cv2.imread(source_path)

            if image is None:
                continue

            transformed = transform(image)
            cv2.imwrite(target_path, transformed)


def low_light(image):
    return cv2.convertScaleAbs(image, alpha=0.7, beta=-40)


def gaussian_blur(image):
    return cv2.GaussianBlur(image, (7, 7), 0)


def rotate_image(image):
    height, width = image.shape[:2]
    matrix = cv2.getRotationMatrix2D(
        (width / 2, height / 2),
        30,
        1.0
    )
    return cv2.warpAffine(image, matrix, (width, height))

In [5]:
# Generate the three transformed datasets

create_transformed_dataset(TEST_DIR, DARK_DIR, low_light)
create_transformed_dataset(TEST_DIR, BLUR_DIR, gaussian_blur)
create_transformed_dataset(TEST_DIR, ROTATE_DIR, rotate_image)

print("Perturbed test datasets created.")

Perturbed test datasets created.


In [6]:
models = {
    model_name: tf.keras.models.load_model(model_path)
    for model_name, model_path in MODEL_PATHS.items()
}

print("Loaded models:")
for model_name, model in models.items():
    print(f"- {model_name}: {model.count_params():,} parameters")

Loaded models:
- Baseline CNN: 11,171,540 parameters
- CNN (Data Augmentation): 11,171,540 parameters
- ResNet50: 23,628,692 parameters
- EfficientNet-B0: 4,075,191 parameters
- ViT: 85,814,036 parameters


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 402 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [7]:
def load_test_dataset(directory):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )


datasets = {
    "Original": load_test_dataset(TEST_DIR),
    "Low-light": load_test_dataset(DARK_DIR),
    "Gaussian blur": load_test_dataset(BLUR_DIR),
    "Rotation": load_test_dataset(ROTATE_DIR),
}

print("Test images per condition:")
for condition, dataset in datasets.items():
    n_images = sum(images.shape[0] for images, _ in dataset)
    print(f"- {condition}: {n_images}")

Found 100 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
Found 100 files belonging to 20 classes.
Test images per condition:
- Original: 100
- Low-light: 100
- Gaussian blur: 100
- Rotation: 100


## Robustness Evaluation

In [8]:
results = []

for model_name, model in models.items():
    print(f"\n===== {model_name} =====")

    for condition, dataset in datasets.items():
        predictions = model.predict(dataset, verbose=0)
        y_pred = np.argmax(predictions, axis=1)
        y_true = np.concatenate([labels.numpy() for _, labels in dataset])

        correct = int(np.sum(y_true == y_pred))
        total = len(y_true)
        accuracy = correct / total

        results.append({
            "Model": model_name,
            "Condition": condition,
            "Correct": correct,
            "Test Size": total,
            "Accuracy (%)": accuracy * 100
        })

        print(f"{condition}: {accuracy * 100:.1f}%")


===== Baseline CNN =====
Original: 77.0%
Low-light: 66.0%
Gaussian blur: 53.0%
Rotation: 42.0%

===== CNN (Data Augmentation) =====
Original: 88.0%
Low-light: 70.0%
Gaussian blur: 70.0%
Rotation: 82.0%

===== ResNet50 =====
Original: 96.0%
Low-light: 93.0%
Gaussian blur: 92.0%
Rotation: 96.0%

===== EfficientNet-B0 =====
Original: 97.0%
Low-light: 98.0%
Gaussian blur: 92.0%
Rotation: 97.0%

===== ViT =====
Original: 94.0%
Low-light: 92.0%
Gaussian blur: 97.0%
Rotation: 95.0%


In [10]:
# Convert results to a DataFrame
results_df = pd.DataFrame(results)

results_df

,Model,Condition,Correct,Test Size,Accuracy (%)
0,Baseline CNN,Original,77,100,77.0
1,Baseline CNN,Low-light,66,100,66.0
2,Baseline CNN,Gaussian blur,53,100,53.0
3,Baseline CNN,Rotation,42,100,42.0
4,CNN (Data Augmentation),Original,88,100,88.0
5,CNN (Data Augmentation),Low-light,70,100,70.0
6,CNN (Data Augmentation),Gaussian blur,70,100,70.0
7,CNN (Data Augmentation),Rotation,82,100,82.0
8,ResNet50,Original,96,100,96.0
9,ResNet50,Low-light,93,100,93.0


In [11]:
accuracy_table = results_df.pivot(
    index="Model",
    columns="Condition",
    values="Accuracy (%)"
)

accuracy_table = accuracy_table[
    ["Original", "Low-light", "Gaussian blur", "Rotation"]
]

accuracy_table.round(1)

Condition,Original,Low-light,Gaussian blur,Rotation
Model,,,,
Baseline CNN,77.0,66.0,53.0,42.0
CNN (Data Augmentation),88.0,70.0,70.0,82.0
EfficientNet-B0,97.0,98.0,92.0,97.0
ResNet50,96.0,93.0,92.0,96.0
ViT,94.0,92.0,97.0,95.0


In [12]:
accuracy_table.round(1).to_csv(
    "robustness_accuracy_table.csv"
)

print("Results saved:")
print("- robustness_accuracy_table.csv")

Results saved:
- robustness_accuracy_table.csv
